# 9장 - 화자 분리

원본 파일: `chap09/speaker_diarization.py`

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pyannote.audio import Pipeline

load_dotenv()

BASE_DIR = (os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())
HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")

/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def diarize_to_rttm(audio_file_path: str, rttm_file_path: str):
    """화자 분리 실행 후 결과를 RTTM 파일로 저장"""
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        token=HUGGING_FACE_TOKEN,
    )

    diarization = pipeline(audio_file_path)

    with open(rttm_file_path, "w", encoding='utf-8') as rttm:
        diarization.write_rttm(rttm)

    return diarization

In [3]:
def rttm_to_grouped_csv(rttm_file_path: str, csv_file_path: str):
    """RTTM 파일을 읽어 같은 화자의 연속 발화끼리 묶은 뒤 CSV로 저장"""
    df_rttm = pd.read_csv(
        rttm_file_path,
        sep=' ',
        header=None,
        names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4'],
    )

    # 발화 시작 + 길이를 더해 끝난 시간(end) 계산
    df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

    # 연속된 발화를 같은 번호로 묶기 위한 number 컬럼
    df_rttm["number"] = None
    df_rttm.at[0, "number"] = 0
    for i in range(1, len(df_rttm)):
        if df_rttm.at[i, "speaker_id"] != df_rttm.at[i - 1, "speaker_id"]:
            df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"] + 1
        else:
            df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"]

    # 같은 화자끼리 묶어서 시작/끝 시간 정리
    df_grouped = df_rttm.groupby("number").agg(
        start=pd.NamedAgg(column='start', aggfunc='min'),
        end=pd.NamedAgg(column='end', aggfunc='max'),
        speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first'),
    )
    df_grouped["duration"] = df_grouped["end"] - df_grouped["start"]
    df_grouped = df_grouped.reset_index(drop=True)

    df_grouped.to_csv(csv_file_path, sep=',', index=False)
    return df_grouped

## 실행 / 테스트

In [4]:
audio_path = os.path.join(BASE_DIR, 'data', '싼기타_비싼기타.mp3')
rttm_path = os.path.join(BASE_DIR, 'data', '싼기타_비싼기타.rttm')
csv_path = os.path.join(BASE_DIR, 'data', '싼기타_비싼기타_rttm.csv')

In [5]:
diarize_to_rttm(audio_path, rttm_path)
df_grouped = rttm_to_grouped_csv(rttm_path, csv_path)
print(df_grouped)

/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/Library/Developer/Comma

      start      end  speaker_id  duration
0     0.993   30.204  SPEAKER_01    29.211
1    32.414   42.708  SPEAKER_00    10.294
2    41.645   44.024  SPEAKER_01     2.379
3    45.813   67.109  SPEAKER_00    21.296
4    67.227   82.786  SPEAKER_01    15.559
5    84.659  102.564  SPEAKER_00    17.905
6   103.492  117.532  SPEAKER_01    14.040
7   119.759  138.676  SPEAKER_00    18.917
8   139.351  168.967  SPEAKER_01    29.616
9   170.907  192.321  SPEAKER_00    21.414
10  192.322  193.689  SPEAKER_01     1.367
11  192.760  193.503  SPEAKER_00     0.743
12  193.823  216.571  SPEAKER_01    22.748
13  218.579  238.120  SPEAKER_00    19.541
14  238.120  238.677  SPEAKER_01     0.557
15  238.188  239.352  SPEAKER_00     1.164
16  239.858  240.651  SPEAKER_01     0.793
17  240.297  241.006  SPEAKER_00     0.709
18  241.040  251.334  SPEAKER_01    10.294
19  253.696  271.550  SPEAKER_00    17.854
20  272.140  304.557  SPEAKER_01    32.417
21  306.970  326.022  SPEAKER_00    19.052
22  326.360